In [1]:
from pathlib import Path
import shutil
import subprocess
import re
from collections import defaultdict
from IPython.display import FileLink, display

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/fall_cnn_c")
WORK.mkdir(parents=True, exist_ok=True)
EXE = WORK / "train_fall_cnn"

EPOCHS = 80          # Đổi thành 2 để chạy thử
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Để None để tự tìm dữ liệu.
# Hoặc điền đường dẫn tới thư mục SisFall của bạn.
DATASET_ROOT = None


def run_live(args):
    command = [str(a) for a in args]

    # Hiện log ngay khi chương trình đang chạy.
    if shutil.which("stdbuf"):
        command = ["stdbuf", "-oL", "-eL"] + command

    with subprocess.Popen(
        command,
        cwd=WORK,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        errors="replace"
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)

        code = process.wait()

    if code:
        raise subprocess.CalledProcessError(code, command)

In [2]:
sources = sorted(INPUT.rglob("train_fall_cnn.c"))
binaries = sorted(INPUT.rglob("train_fall_cnn"))

if sources:
    source = WORK / "train_fall_cnn.c"
    shutil.copy2(sources[0], source)

    print("Biên dịch:", sources[0], flush=True)

    run_live([
        "gcc", "-O3", "-std=c11", "-Wall", "-Wextra",
        source, "-lm", "-o", EXE
    ])

elif binaries:
    uploaded = binaries[0]

    with uploaded.open("rb") as f:
        magic = f.read(4)

    if magic == b"\x7fELF":
        shutil.copy2(uploaded, EXE)
        print("Dùng chương trình Linux:", uploaded, flush=True)
    else:
        # Hỗ trợ mã nguồn C được upload thiếu đuôi .c.
        source = WORK / "train_fall_cnn.c"
        shutil.copy2(uploaded, source)

        run_live([
            "gcc", "-O3", "-std=c11", "-Wall", "-Wextra",
            source, "-lm", "-o", EXE
        ])

else:
    raise FileNotFoundError(
        "Hãy thêm dataset chứa train_fall_cnn hoặc train_fall_cnn.c"
    )

EXE.chmod(0o755)

print("\nChạy self-test...", flush=True)
run_live([EXE, "--self-test"])

Biên dịch: /kaggle/input/datasets/trnhvnnghim/train-fall-cnn/train_fall_cnn.c

Chạy self-test...
Epoch   1 | lr 0.0010000 | train loss 0.693052 acc 50.00% | val loss 0.684341 acc 100.00%
Epoch   2 | lr 0.0010000 | train loss 0.322298 acc 98.44% | val loss 0.679353 acc 100.00%
Epoch   3 | lr 0.0010000 | train loss 0.199762 acc 100.00% | val loss 0.670551 acc 100.00%
Epoch   4 | lr 0.0010000 | train loss 0.153580 acc 100.00% | val loss 0.658989 acc 100.00%
Epoch   5 | lr 0.0010000 | train loss 0.108966 acc 100.00% | val loss 0.643403 acc 100.00%
Epoch   6 | lr 0.0010000 | train loss 0.099244 acc 100.00% | val loss 0.626336 acc 100.00%
Epoch   7 | lr 0.0010000 | train loss 0.067727 acc 100.00% | val loss 0.608596 acc 100.00%
Epoch   8 | lr 0.0010000 | train loss 0.070361 acc 100.00% | val loss 0.591003 acc 100.00%
Epoch   9 | lr 0.0010000 | train loss 0.067442 acc 100.00% | val loss 0.573599 acc 100.00%
Epoch  10 | lr 0.0010000 | train loss 0.067884 acc 100.00% | val loss 0.555483 acc 100

In [3]:
if DATASET_ROOT is None:
    groups = defaultdict(list)

    for f in INPUT.rglob("*"):
        if not f.is_file():
            continue

        if f.suffix.lower() != ".txt":
            continue

        # Tên bản ghi SisFall, ví dụ D01_SA01_R01.txt.
        if not re.match(r"^[DF]\d{2}_", f.name, re.IGNORECASE):
            continue

        root = next(
            (
                p for p in f.parents
                if p != INPUT and "sisfall" in p.name.lower()
            ),
            f.parent.parent
        )

        groups[root].append(f)

    if not groups:
        raise FileNotFoundError(
            "Không tìm thấy dữ liệu SisFall raw .txt. "
            "Hãy thêm dataset SisFall đã giải nén vào Input."
        )

    candidates = sorted(
        groups,
        key=lambda p: (
            "sisfall-dataset-fall-detection" in str(p).lower(),
            len(groups[p])
        ),
        reverse=True
    )

    print("Các bộ dữ liệu tìm thấy:")
    for root in candidates:
        print(f"  {root}: {len(groups[root])} tệp")

    dataset = candidates[0]

else:
    dataset = Path(DATASET_ROOT)

assert dataset.is_dir(), f"Đường dẫn không tồn tại: {dataset}"

print("\nDữ liệu được chọn:", dataset)

Các bộ dữ liệu tìm thấy:
  /kaggle/input/datasets/kushajm/sisfall-dataset-fall-detection/SisFall_dataset: 4505 tệp

Dữ liệu được chọn: /kaggle/input/datasets/kushajm/sisfall-dataset-fall-detection/SisFall_dataset


In [4]:
run_live([
    EXE,
    dataset,
    EPOCHS,
    BATCH_SIZE,
    LEARNING_RATE
])

print("\nHoàn tất. Kết quả:", WORK)

Loading SisFall from: /kaggle/input/datasets/kushajm/sisfall-dataset-fall-detection/SisFall_dataset
Loaded 4505 usable recordings.
Subjects: 38 | train samples=2871 val=415 test=1219
Training after augmentation/balance: 6692 samples (ADL=3346 Fall=3346)
Epoch   1 | lr 0.0010000 | train loss 0.347586 acc 85.76% | val loss 0.723384 acc 63.86%
Epoch   2 | lr 0.0010000 | train loss 0.151838 acc 94.08% | val loss 0.573465 acc 67.95%
Epoch   3 | lr 0.0010000 | train loss 0.074314 acc 98.03% | val loss 0.047451 acc 99.52%
Epoch   4 | lr 0.0010000 | train loss 0.050617 acc 98.61% | val loss 0.007892 acc 100.00%
Epoch   5 | lr 0.0010000 | train loss 0.037090 acc 98.83% | val loss 0.009936 acc 99.76%
Epoch   6 | lr 0.0010000 | train loss 0.026547 acc 99.30% | val loss 0.007477 acc 99.76%
Epoch   7 | lr 0.0010000 | train loss 0.018521 acc 99.48% | val loss 0.004724 acc 100.00%
Epoch   8 | lr 0.0010000 | train loss 0.020738 acc 99.37% | val loss 0.005605 acc 99.52%
Epoch   9 | lr 0.0010000 | train

In [5]:
import os

shutil.make_archive(
    "/kaggle/working/fall_cnn_results",
    "zip",
    root_dir=WORK
)

os.chdir("/kaggle/working")
display(FileLink("fall_cnn_results.zip"))

/kaggle/working/fall_cnn_results.zip